In [ ]:
# Stage 06 bootstrap and controls.
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import shutil
import subprocess
import sys

import yaml

RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"
PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
PROJECT_REPO_COMMIT = "279a964fddca73dbe541c10145df74a2b325a78c"
PROJECT_DIR = Path("/content/lst_models")

STAGE_NAME = "06_ian_final_progress_record"
SCOPE = "progress_record_measure_only"
HOLDOUT_TEST_CONTACT = False
NEW_SCORING_EVENTS = 0
NO_FINAL_MODEL_SELECTED = True
OFFICIAL_VALIDATION_FOR_SELECTION = False

RUN_STAGE06 = False
RUN_STAGE06_DRIVE_SYNC = True
RUN_STAGE06_DRIVE_BACKUP = True
RUN_ARTIFACT_DISPLAY = False

STAGE_CONFIG_RELATIVE = Path("configs/stages/06_ian_final_progress_record.yaml")
STAGE_PROTOCOL_RELATIVE = Path("docs/protocols/06_ian_final_progress_record_protocol.md")
STAGE_ENTRYPOINT_RELATIVE = Path("src/lst_models/stages/ian_final_progress_record.py")
NOTEBOOK_RELATIVE = Path("notebooks/06_ian_final_progress_record_colab.ipynb")
OUTPUT_DIR = Path("/content/lst_models_results/06_ian_final_progress_record")
STAGE06_DRIVE_RESULT_PATH_PARTS = ["lst_models", "results", STAGE_NAME]

UPSTREAM_RUNTIME_RUN_DIRS = {
    "00_data_split_label_freeze": Path("/content/lst_models_results/00_data_split_label_freeze/20260610_051705_347450"),
    "01_feature_window_search": Path("/content/lst_models_results/01_feature_window_search/20260610_075002"),
    "02_model_hpo_train_inner": Path("/content/lst_models_results/02_model_hpo_train_inner/20260610_082130_797479"),
    "03_frozen_validation_readout": Path("/content/lst_models_results/03_frozen_validation_readout/20260610_133305_716174"),
    "04_diagnostics_ablation": Path("/content/lst_models_results/04_diagnostics_ablation/20260619_082125_765984"),
    "05_thesis_synthesis": Path("/content/lst_models_results/05_thesis_synthesis/20260619_090454_562658"),
    "v2_1_guarded_walkforward_readout": Path("/content/lst_models_results/v2_1_guarded_walkforward_readout/20260618_063559_889276"),
}
UPSTREAM_DRIVE_PATH_PARTS = {
    name: ["lst_models", "results", name, run_dir.name]
    for name, run_dir in UPSTREAM_RUNTIME_RUN_DIRS.items()
}
REQUIRED_UPSTREAM_RECORD_FILES = ["artifact_inventory.csv", "run_manifest.json"]


def run_command(args: list[str], *, cwd: Path | None = None) -> str:
    completed = subprocess.run(
        args,
        cwd=str(cwd) if cwd else None,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    return completed.stdout.strip()


def project_root_from_runtime() -> Path:
    cwd = Path.cwd()
    if (cwd / "src" / "lst_models").exists() and (cwd / "configs").exists():
        return cwd
    if PROJECT_DIR.exists():
        return PROJECT_DIR
    return cwd


def bootstrap_project() -> Path:
    if PROJECT_BOOTSTRAP_MODE != "github_commit":
        raise ValueError(f"Unsupported PROJECT_BOOTSTRAP_MODE: {PROJECT_BOOTSTRAP_MODE}")
    if not PROJECT_REPO_COMMIT or PROJECT_REPO_COMMIT.startswith("<"):
        raise ValueError("PROJECT_REPO_COMMIT must be an exact commit hash before Colab execution")
    if RUN_PROJECT_BOOTSTRAP:
        if not PROJECT_DIR.exists():
            run_command(["git", "clone", PROJECT_REPO_URL, str(PROJECT_DIR)])
        else:
            run_command(["git", "fetch", "--all", "--tags"], cwd=PROJECT_DIR)
        run_command(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_DIR)
        resolved = run_command(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR)
        if resolved != PROJECT_REPO_COMMIT:
            raise RuntimeError(f"Resolved {resolved} but expected {PROJECT_REPO_COMMIT}")
        root = PROJECT_DIR
    else:
        root = project_root_from_runtime()
    required = [
        STAGE_CONFIG_RELATIVE,
        STAGE_PROTOCOL_RELATIVE,
        STAGE_ENTRYPOINT_RELATIVE,
        NOTEBOOK_RELATIVE,
        Path("src/lst_models"),
        Path("configs"),
        Path("docs/protocols"),
    ]
    missing = [str(root / rel) for rel in required if not (root / rel).exists()]
    if missing:
        raise FileNotFoundError("Stage 06 bootstrap target is missing required files: " + "; ".join(missing))
    src_dir = root / "src"
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))
    return root


PROJECT_ROOT = bootstrap_project()
STAGE_CONFIG_PATH = PROJECT_ROOT / STAGE_CONFIG_RELATIVE
NOTEBOOK_PATH = PROJECT_ROOT / NOTEBOOK_RELATIVE
print("project_root", PROJECT_ROOT)
print("project_commit", PROJECT_REPO_COMMIT)
print("stage", STAGE_NAME)


# Stage 06 - Ian Final Progress Record

This notebook is the user-facing entry point for the Stage 06 progress record. It inventories the frozen upstream stage artifacts, writes the Ian/Lan requirement mapping, and saves the Stage 06 record bundle to Drive.

Scope guard:

- No model fitting, threshold search, or new scoring occurs here.
- No holdout/test rows are read, transformed, windowed, scored, summarized, or used.
- The record is a validation-only and guarded historically-contacted progress record.
- Upstream artifacts are fetched only by exact Drive path parts and exact run ids.


In [ ]:
# Load Stage 06 config and inject runtime paths before assertions.
with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    stage06_config = yaml.safe_load(handle)

if stage06_config["stage_name"] != STAGE_NAME:
    raise ValueError(f"Expected stage_name {STAGE_NAME}, got {stage06_config['stage_name']}")
if stage06_config["scope"] != SCOPE:
    raise ValueError(f"Expected scope {SCOPE}, got {stage06_config['scope']}")
if stage06_config["holdout_test_contact"] is not HOLDOUT_TEST_CONTACT:
    raise ValueError("Stage 06 must keep holdout_test_contact=false")
if int(stage06_config["new_scoring_events"]) != NEW_SCORING_EVENTS:
    raise ValueError("Stage 06 must keep new_scoring_events=0")
if stage06_config["no_final_model_selected"] is not NO_FINAL_MODEL_SELECTED:
    raise ValueError("Stage 06 must not select a model")
if stage06_config["official_validation_for_selection"] is not OFFICIAL_VALIDATION_FOR_SELECTION:
    raise ValueError("Stage 06 must not use official validation for selection")

stage_inputs = stage06_config["inputs"]
stage_outputs = stage06_config["outputs"]
stage_inputs["notebook_path"] = str(NOTEBOOK_RELATIVE.as_posix())
stage_outputs["output_dir"] = str(OUTPUT_DIR)

upstream_by_name = {str(run["stage_name"]): run for run in stage_inputs["upstream_runs"]}
if set(upstream_by_name) != set(UPSTREAM_RUNTIME_RUN_DIRS):
    raise ValueError(f"Unexpected Stage 06 upstream set: {sorted(upstream_by_name)}")
for stage_name, run_dir in UPSTREAM_RUNTIME_RUN_DIRS.items():
    run = upstream_by_name[stage_name]
    if str(run["run_id"]) != run_dir.name:
        raise ValueError(f"Run id mismatch for {stage_name}: {run['run_id']} vs {run_dir.name}")
    run["runtime_run_dir"] = str(run_dir)
    run["drive_path_parts"] = UPSTREAM_DRIVE_PATH_PARTS[stage_name]

print("output_dir", stage_outputs["output_dir"])
print("upstream_runs", len(stage_inputs["upstream_runs"]))


In [ ]:
# Optional exact-run Drive sync for upstream manifests and inventories.
def quote_drive_query_value(value: str) -> str:
    return value.replace(chr(92), chr(92) * 2).replace("\'", chr(92) + "\'")


def get_drive_service():
    from google.colab import auth
    from googleapiclient.discovery import build

    auth.authenticate_user()
    return build("drive", "v3")


def find_unique_drive_child(service, parent_id: str, name: str, *, mime_type: str | None = None):
    clauses = [
        f"'{parent_id}' in parents",
        "trashed = false",
        f"name = '{quote_drive_query_value(name)}'",
    ]
    if mime_type is not None:
        clauses.append(f"mimeType = '{quote_drive_query_value(mime_type)}'")
    response = service.files().list(
        q=" and ".join(clauses),
        fields="files(id,name,mimeType,size)",
        pageSize=10,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()
    files = response.get("files", [])
    if len(files) > 1:
        raise RuntimeError(f"Duplicate Drive item named {name!r} under parent {parent_id}")
    return files[0] if files else None


def ensure_drive_folder(service, parent_id: str, name: str) -> str:
    folder = find_unique_drive_child(service, parent_id, name, mime_type="application/vnd.google-apps.folder")
    if folder is not None:
        return folder["id"]
    created = service.files().create(
        body={"name": name, "parents": [parent_id], "mimeType": "application/vnd.google-apps.folder"},
        fields="id",
        supportsAllDrives=True,
    ).execute()
    return created["id"]


def resolve_drive_path(service, path_parts: list[str], *, create: bool = False) -> str:
    parent_id = "root"
    for part in path_parts:
        if create:
            parent_id = ensure_drive_folder(service, parent_id, str(part))
        else:
            folder = find_unique_drive_child(
                service,
                parent_id,
                str(part),
                mime_type="application/vnd.google-apps.folder",
            )
            if folder is None:
                raise FileNotFoundError(f"Drive folder path is missing at part {part!r}: {path_parts}")
            parent_id = folder["id"]
    return parent_id


def download_drive_file(service, parent_id: str, file_name: str, target_path: Path) -> None:
    from googleapiclient.http import MediaIoBaseDownload
    import io

    item = find_unique_drive_child(service, parent_id, file_name)
    if item is None:
        raise FileNotFoundError(f"Missing Drive file {file_name!r} under folder id {parent_id}")
    target_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=item["id"], supportsAllDrives=True)
    with target_path.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()


def missing_upstream_record_files() -> list[tuple[str, list[str], str]]:
    missing_items = []
    for run in stage_inputs["upstream_runs"]:
        local_dir = Path(str(run["runtime_run_dir"]))
        missing = [name for name in REQUIRED_UPSTREAM_RECORD_FILES if not (local_dir / name).exists()]
        if missing:
            missing_items.append((str(run["stage_name"]), missing, str(local_dir)))
    return missing_items


def format_missing_upstream_records(missing_items: list[tuple[str, list[str], str]]) -> str:
    return "; ".join(
        f"{stage}: missing {names} under {run_dir}"
        for stage, names, run_dir in missing_items
    )


def fetch_missing_upstream_records_from_drive() -> None:
    missing_before = missing_upstream_record_files()
    if not missing_before:
        print("all upstream record files already present")
        return
    if not RUN_STAGE06_DRIVE_SYNC:
        message = format_missing_upstream_records(missing_before)
        if RUN_STAGE06:
            raise FileNotFoundError("Stage 06 upstream record files are missing: " + message)
        print("upstream record files pending Drive fetch", missing_before)
        return

    service = get_drive_service()
    for run in stage_inputs["upstream_runs"]:
        local_dir = Path(str(run["runtime_run_dir"]))
        missing = [name for name in REQUIRED_UPSTREAM_RECORD_FILES if not (local_dir / name).exists()]
        if not missing:
            continue
        folder_id = resolve_drive_path(service, [str(part) for part in run["drive_path_parts"]], create=False)
        for file_name in missing:
            download_drive_file(service, folder_id, file_name, local_dir / file_name)
        print("fetched", str(run["stage_name"]), missing)

    remaining = missing_upstream_record_files()
    if remaining:
        message = format_missing_upstream_records(remaining)
        if RUN_STAGE06:
            raise FileNotFoundError("Stage 06 upstream record files are still missing after Drive sync: " + message)
        print("upstream record files still pending Drive fetch", remaining)


fetch_missing_upstream_records_from_drive()


In [ ]:
# Run Stage 06. Keep RUN_STAGE06=False until the upstream artifacts are present.
if RUN_STAGE06:
    from lst_models.stages.ian_final_progress_record import run_stage

    result = run_stage(stage06_config)
    STAGE06_RUN_DIR = Path(result.run_dir)
    STAGE06_RUN_ID = STAGE06_RUN_DIR.name
    print("stage_run_id", STAGE06_RUN_ID)
    print("stage_run_dir", STAGE06_RUN_DIR)
else:
    result = None
    STAGE06_RUN_DIR = None
    STAGE06_RUN_ID = None
    print("RUN_STAGE06=False; Stage 06 was not executed in this notebook session.")


In [ ]:
# Durable Drive save cell. This cell must remain immediately after the run_stage cell.
from lst_models.stages.ian_final_progress_record import REQUIRED_STAGE06_ARTIFACTS


def upload_or_update_file(service, parent_id: str, local_path: Path) -> str:
    from googleapiclient.http import MediaFileUpload

    existing = find_unique_drive_child(service, parent_id, local_path.name)
    media = MediaFileUpload(str(local_path), resumable=True)
    if existing is None:
        created = service.files().create(
            body={"name": local_path.name, "parents": [parent_id]},
            media_body=media,
            fields="id",
            supportsAllDrives=True,
        ).execute()
        return created["id"]
    updated = service.files().update(
        fileId=existing["id"],
        media_body=media,
        fields="id",
        supportsAllDrives=True,
    ).execute()
    return updated["id"]


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def validate_stage06_run_folder(run_dir: Path) -> dict:
    missing = [name for name in REQUIRED_STAGE06_ARTIFACTS if not (run_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing Stage 06 required artifacts {missing} under {run_dir}")
    manifest_path = run_dir / "run_manifest.json"
    progress_record_path = run_dir / "06_progress_record.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    progress_record = json.loads(progress_record_path.read_text(encoding="utf-8"))
    if progress_record.get("n_pending_drive_fetch") != 0:
        raise ValueError(
            "Stage 06 backup refused because upstream artifacts are still pending Drive fetch: "
            f"n_pending_drive_fetch={progress_record.get('n_pending_drive_fetch')}"
        )
    expected_pairs = {
        "stage_name": STAGE_NAME,
        "scope": SCOPE,
        "holdout_test_contact": HOLDOUT_TEST_CONTACT,
        "new_scoring_events": NEW_SCORING_EVENTS,
        "official_validation_for_selection": OFFICIAL_VALIDATION_FOR_SELECTION,
        "no_final_model_selected": NO_FINAL_MODEL_SELECTED,
    }
    for key, expected in expected_pairs.items():
        if manifest.get(key) != expected:
            raise ValueError(f"Stage 06 manifest field {key!r} is {manifest.get(key)!r}; expected {expected!r}")
        if progress_record.get(key) != expected:
            raise ValueError(
                f"Stage 06 progress record field {key!r} is {progress_record.get(key)!r}; expected {expected!r}"
            )
    if manifest.get("notebook_sha256") is None:
        raise ValueError("Stage 06 manifest notebook_sha256 is missing; check inputs.notebook_path")
    return manifest


def backup_stage06_results_to_drive(run_dir: Path) -> dict:
    manifest = validate_stage06_run_folder(run_dir)
    service = get_drive_service()
    drive_path_parts = [*STAGE06_DRIVE_RESULT_PATH_PARTS, run_dir.name]
    target_folder_id = resolve_drive_path(service, drive_path_parts, create=True)
    uploaded_files = []
    for file_name in REQUIRED_STAGE06_ARTIFACTS:
        local_path = run_dir / file_name
        file_id = upload_or_update_file(service, target_folder_id, local_path)
        uploaded_files.append({
            "file_name": file_name,
            "drive_file_id": file_id,
            "uploaded_byte_size": local_path.stat().st_size,
        })
    backup_manifest_path = run_dir / "drive_backup_manifest.json"
    backup_manifest = {
        "stage_name": STAGE_NAME,
        "run_id": run_dir.name,
        "local_output_dir": str(run_dir),
        "drive_path_parts": drive_path_parts,
        "drive_folder_id": target_folder_id,
        "uploaded_files": [
            *uploaded_files,
            {
                "file_name": backup_manifest_path.name,
                "drive_file_id": None,
                "uploaded_byte_size": None,
            },
        ],
        "source_run_manifest": manifest,
        "sync_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_json(backup_manifest_path, backup_manifest)
    self_file_id = upload_or_update_file(service, target_folder_id, backup_manifest_path)
    backup_manifest["uploaded_files"][-1]["drive_file_id"] = self_file_id
    write_json(backup_manifest_path, backup_manifest)
    upload_or_update_file(service, target_folder_id, backup_manifest_path)
    print("stage_run_id", run_dir.name)
    print("drive_path", "/".join(drive_path_parts))
    print("drive_folder_id", target_folder_id)
    return backup_manifest


if RUN_STAGE06 and RUN_STAGE06_DRIVE_BACKUP:
    stage06_drive_backup_manifest = backup_stage06_results_to_drive(STAGE06_RUN_DIR)
elif RUN_STAGE06:
    validate_stage06_run_folder(STAGE06_RUN_DIR)
    print("RUN_STAGE06_DRIVE_BACKUP=False; local Stage 06 artifacts validated at", STAGE06_RUN_DIR)
else:
    stage06_drive_backup_manifest = None
    print("Stage 06 save skipped because RUN_STAGE06=False.")


In [ ]:
# Optional compact artifact display after a run.
if RUN_ARTIFACT_DISPLAY and RUN_STAGE06:
    import pandas as pd

    display(pd.read_csv(STAGE06_RUN_DIR / "06_reproducibility_inventory.csv"))
    display(pd.read_csv(STAGE06_RUN_DIR / "06_ian_requirement_mapping.csv"))
    progress_record = json.loads((STAGE06_RUN_DIR / "06_progress_record.json").read_text(encoding="utf-8"))
    print(json.dumps(progress_record, indent=2, sort_keys=True))
else:
    print("RUN_ARTIFACT_DISPLAY=False or no Stage 06 run in this session.")


## Interpretation Guard

Use the Stage 06 artifacts as a route-closing progress record only. They do not select a model, do not introduce new scoring, and do not change the validation-only claim boundary recorded in `paper/outline_and_claims.md`.
